# Fine-tune Football YOLO11x

Fine-tunes `best-finetune.pt` on the Roboflow football dataset.

Before running: select **Runtime → Change runtime type → T4 GPU**, then add the Colab secret `ROBOFLOW_API_KEY` and enable notebook access.


In [ ]:
!pip install -q -U ultralytics roboflow pyyaml


In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU: Runtime → Change runtime type → T4 GPU"
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/FootballTraining')
RUNS_DIR = DRIVE_ROOT / 'runs'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)


## Model

The checkpoint is loaded directly from `MyDrive/Futvar/best-finetune.pt`.


In [ ]:
from pathlib import Path

DRIVE_MODEL = Path('/content/drive/MyDrive/Futvar/best-finetune.pt')
SOURCE_MODEL = DRIVE_MODEL
BASE_MODEL = DRIVE_MODEL

assert DRIVE_MODEL.exists(), f'Model was not found: {DRIVE_MODEL}'
assert DRIVE_MODEL.stat().st_size > 100_000_000, 'Checkpoint is incomplete or corrupted.'

print('Model loaded from Google Drive:', DRIVE_MODEL)
print('Size MB:', round(DRIVE_MODEL.stat().st_size / 1024**2, 2))


In [ ]:
from ultralytics import YOLO

test_model = YOLO(str(SOURCE_MODEL))
print('Checkpoint loaded successfully.')
print('Classes:', test_model.names)


In [ ]:
BASE_MODEL = DRIVE_MODEL
print('Training checkpoint:', BASE_MODEL)


## Dataset

The API key is read from Colab Secrets and is never printed in the notebook.


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

api_key = userdata.get('ROBOFLOW_API_KEY')
assert api_key, 'Add ROBOFLOW_API_KEY in the Colab Secrets panel.'

rf = Roboflow(api_key=api_key)
project = rf.workspace('ahmed-maged-my0pv').project('futvar-football-players-detection-dataset-ounwf')
dataset = project.version(1).download('yolov8', location='/content/futvar_dataset')

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / 'data.yaml'
assert DATA_YAML.exists()
print(DATA_YAML)


In [ ]:
import yaml

with open(DATA_YAML, encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

def names_to_list(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names)

model = YOLO(str(BASE_MODEL))
model_names = names_to_list(model.names)
dataset_names = names_to_list(data_config['names'])
expected = ['ball', 'goalkeeper', 'player', 'referee']

print('Model:  ', model_names)
print('Dataset:', dataset_names)
assert model_names == expected, f'Unexpected model classes: {model_names}'
assert dataset_names == expected, f'Unexpected dataset classes: {dataset_names}'
print('Class order is compatible.')


## Baseline evaluation


In [ ]:
baseline = model.val(
    data=str(DATA_YAML), split='test', imgsz=640, batch=4,
    device=0, workers=2, plots=True,
    project=str(DRIVE_ROOT / 'evaluations'), name='before-finetune', exist_ok=True,
)
print('Before fine-tuning — mAP50:', baseline.box.map50, 'mAP50-95:', baseline.box.map)


## Fine-tuning

Checkpoints are written to Drive. If Colab disconnects, run the resume cell below.


In [ ]:
RUN_NAME = 'football-yolo11x-finetuned'
model = YOLO(str(BASE_MODEL))

results = model.train(
    data=str(DATA_YAML),
    epochs=60,
    patience=15,
    imgsz=640,
    batch=-1,
    device=0,
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    amp=True,
    cos_lr=True,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    save_period=5,
    plots=True,
    seed=42,
)


## Resume only after an interruption

Do not run this cell after training finishes normally.


In [ ]:
LAST = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
assert LAST.exists(), f'No checkpoint found: {LAST}'
YOLO(str(LAST)).train(resume=True)


## Final evaluation


In [ ]:
BEST = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
assert BEST.exists(), f'No final model found: {BEST}'

final_model = YOLO(str(BEST))
final = final_model.val(
    data=str(DATA_YAML), split='test', imgsz=640, batch=4,
    device=0, workers=2, plots=True,
    project=str(DRIVE_ROOT / 'evaluations'), name='after-finetune', exist_ok=True,
)

print('Before mAP50:', baseline.box.map50, 'mAP50-95:', baseline.box.map)
print('After  mAP50:', final.box.map50, 'mAP50-95:', final.box.map)
print('Best model:', BEST)


In [ ]:
from IPython.display import display, Image

test_images = list((DATASET_DIR / 'test' / 'images').glob('*'))[:8]
final_model.predict(
    source=[str(p) for p in test_images], imgsz=640, conf=0.20,
    device=0, save=True, project=str(DRIVE_ROOT / 'predictions'),
    name='fine-tuned-samples', exist_ok=True,
)

for image_path in list((DRIVE_ROOT / 'predictions' / 'fine-tuned-samples').glob('*'))[:8]:
    display(Image(filename=str(image_path), width=900))
